In [11]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Polygon
import scipy.io as sio
from matplotlib.colors import LinearSegmentedColormap
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.gridspec import GridSpec

from monsoonbench.spatial.regions import get_india_outline

import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from plot_config import params, contourLevels, colormap, savefig_format, SMALL_SIZE, MEDIUM_SIZE, LARGE_SIZE

# Apply plot settings
plt.rcParams.update(params)


# Your plotting code here...

🎨 Scientific plotting configuration loaded
   Default save format: png
   Contour levels: 100
   Colormap: bwr


In [12]:
# Configuration parameters
model_str = ['Climatology', 'IFS', 'AIFS', 'FuXi', 'Graphcast', 'GenCast', 'FuXi S2S', 'NGCM']
#dummy_str = ['GenCast']
yr_ar = np.arange(2019, 2025)  # [2019:2024]


In [13]:

polygon_col = np.array([49, 130, 189]) / 256
map_col = 0.3*np.array([1, 1, 1])

# Define Core Monsoon Zone polygon
polygon1_lon = np.array([86, 74, 74, 70, 70, 82, 82, 86, 86])
polygon1_lat = np.array([18, 18, 22, 22, 30, 30, 26, 26, 18])


In [225]:
# def get_india_outline():
#     """
#     Get India outline coordinates. First try to load from shapefile, 
#     then fall back to simplified coordinates.
#     """
#     try:
#         # Try to load India shapefile - use the specified path first
#         shapefile_paths = [
#             '../fig_data/ind_map_shapefile/india_shapefile.shp',
#             'india_shapefile.shp'
#         ]
        
#         for path in shapefile_paths:
#             try:
#                 import geopandas as gpd
#                 india_gdf = gpd.read_file(path)
#                 # Extract boundary coordinates
#                 boundaries = []
#                 for geom in india_gdf.geometry:
#                     if hasattr(geom, 'exterior'):
#                         coords = list(geom.exterior.coords)
#                         lon_coords = [coord[0] for coord in coords]
#                         lat_coords = [coord[1] for coord in coords]
#                         boundaries.append((lon_coords, lat_coords))
#                     elif hasattr(geom, 'geoms'):
#                         for sub_geom in geom.geoms:
#                             if hasattr(sub_geom, 'exterior'):
#                                 coords = list(sub_geom.exterior.coords)
#                                 lon_coords = [coord[0] for coord in coords]
#                                 lat_coords = [coord[1] for coord in coords]
#                                 boundaries.append((lon_coords, lat_coords))
#                 return boundaries
#             except:
#                 continue
#     except:
#         pass
    
#     return [(india_lon, india_lat)]

# print("India outline function defined")

In [14]:
import pandas as pd
df = pd.read_csv("outputs/fig_7_through_12_data_recreation.csv")
df = df.loc[df["horizon"] == 15]
df.head(10)

,Unnamed: 0,lat,lon,Mean MAE,model,horizon,Miss Rate,False Alarm Rate
0,0,8.0,76.0,4.500000,Climatology,15,36.0,19.5
1,1,8.0,80.0,30.500000,Climatology,15,100.0,1.2
2,2,12.0,76.0,5.333333,Climatology,15,37.5,18.6
3,3,12.0,80.0,7.750000,Climatology,15,62.5,14.8
4,4,16.0,72.0,4.166667,Climatology,15,25.0,15.6
5,5,16.0,76.0,4.833333,Climatology,15,33.3,25.0
6,6,16.0,80.0,6.000000,Climatology,15,38.5,22.6
7,7,16.0,84.0,8.666667,Climatology,15,59.3,25.9
8,8,20.0,68.0,8.600000,Climatology,15,58.3,10.7
9,9,20.0,72.0,5.600000,Climatology,15,52.0,12.5


In [28]:
# Load data from MAT files
# Define file paths
file_15day = '/Users/charlieeden/Downloads/mae_far_mr_15_day_2019_2024 (2).mat'
print("Loading 15-day data...")
data_15 = sio.loadmat(file_15day)

    
    # Extract coordinate data (assuming they're in the MAT files)
    # Check what variables are available in the MAT files
print("Available variables in 15-day file:", [key for key in data_15.keys() if not key.startswith('__')])
    
    # Extract coordinates - adjust variable names as needed based on your MAT file structure
if 'lon' in data_15:
    lons = data_15['lon'].flatten()
    lats = data_15['lat'].flatten()
else:
    # If coordinates not found, use default range - you may need to adjust this
    print("Warning: Coordinates not found in MAT file, using default range")
    lon = np.arange(70, 101, 4)
    lat = np.arange(8, 39, 4)

    # Extract MAE data - adjust variable names as needed

mae_avg = data_15['mae_avg']
mae_cmz = data_15['mae_cmz_mean']
std_er = data_15['std_er']
far = data_15['false_alarm']
far_cmz = data_15['far_cmz_mean']
mr = data_15['miss_rate']
mr_cmz = data_15['mr_cmz_mean']

    

Loading 15-day data...
Available variables in 15-day file: ['csi_cmz_mean', 'csi_yr', 'ets_cmz_mean', 'ets_yr', 'false_alarm', 'far_cmz_mean', 'lat', 'lon', 'mae_avg', 'mae_cmz_mean', 'miss_rate', 'model_str', 'mr_cmz_mean', 'std_er']


In [36]:
lats

array([ 8, 12, 16, 20, 24, 28, 32, 36], dtype=uint8)

In [37]:
np.arange(8,37,4)

array([ 8, 12, 16, 20, 24, 28, 32, 36])

In [29]:
def reformat_csv_as_grids(df, metric):
    
    model_vals = df['model'].unique()

    lat_to_idx = {v: i for i, v in enumerate(lats)}
    lon_to_idx = {v: i for i, v in enumerate(lons)}

    # output dict
    data_dict = {}

    for model in model_vals:
        sub = df[df['model'] == model]

        grid = np.full((len(lats), len(lons)), np.nan)

        for _, row in sub.iterrows():
            i = lat_to_idx[row['lat']]
            j = lon_to_idx[row['lon']]
            grid[i, j] = row[metric]

        data_dict[model] = grid

    ordered_dict = {m: data_dict[m] for m in model_str if m in data_dict}
    ordered_dict["None"] = np.full((len(lats), len(lons)), np.nan)

    return ordered_dict


In [33]:
df

,Unnamed: 0,lat,lon,Mean MAE,model,horizon,Miss Rate,False Alarm Rate
0,0,8.0,76.0,4.500000,Climatology,15,36.0,19.5
1,1,8.0,80.0,30.500000,Climatology,15,100.0,1.2
2,2,12.0,76.0,5.333333,Climatology,15,37.5,18.6
3,3,12.0,80.0,7.750000,Climatology,15,62.5,14.8
4,4,16.0,72.0,4.166667,Climatology,15,25.0,15.6
...,...,...,...,...,...,...,...,...
275,30,32.0,72.0,19.395833,NGCM,15,0.0,33.6
276,31,32.0,76.0,9.322222,NGCM,15,55.0,16.5
277,32,32.0,80.0,5.750000,NGCM,15,88.0,2.6
278,33,36.0,72.0,NaN,NGCM,15,100.0,0.0


In [30]:
mae_avg = reformat_csv_as_grids(df=df, metric="Mean MAE")
mae_avg = np.stack([val for key, val in mae_avg.items()], axis=0)

far = reformat_csv_as_grids(df=df, metric="False Alarm Rate")
far = np.stack([val for key, val in far.items()], axis=0)


mr = reformat_csv_as_grids(df=df, metric="Miss Rate")
mr = np.stack([val for key, val in mr.items()], axis=0)



In [32]:
mae_avg[0]

array([[        nan,         nan,  4.5       , 30.5       ,         nan,
                nan,         nan,         nan,         nan],
       [        nan,         nan,  5.33333333,  7.75      ,         nan,
                nan,         nan,         nan,         nan],
       [        nan,  4.16666667,  4.83333333,  6.        ,  8.66666667,
                nan,         nan,         nan,         nan],
       [ 8.6       ,  5.6       ,  8.16666667,  4.5       ,  4.        ,
         7.33333333,         nan,         nan,         nan],
       [ 9.        ,  6.6       ,  2.66666667,  5.4       ,  6.66666667,
         9.16666667,  1.16666667,  4.        ,         nan],
       [11.25      ,  9.        ,  7.83333333,  2.6       ,  6.66666667,
         3.        ,  3.5       ,  4.5       ,         nan],
       [        nan, 29.        , 17.        ,  9.66666667,         nan,
                nan,         nan,         nan,         nan],
       [        nan, 16.        , 14.6       ,         nan,   